In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm



In [9]:
#helper codes
def onehot_encode(seq, max_len):
    mat = np.zeros((20, max_len), dtype=np.float32)
    for i, aa in enumerate(seq[:max_len]):
        if aa in AA_TO_INDEX:
            mat[AA_TO_INDEX[aa], i] = 1.0
    return mat

In [10]:
# ------------------------- Helper -------------------------
AA_TO_INDEX = {aa: idx for idx, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}

# class ProteinDataset(Dataset):
#     def __init__(self, sequences, labels):
#         self.sequences = sequences
#         self.labels = labels
#         self.max_len = max_len

#     def __len__(self):
#         return len(self.sequences)

#     # def __getitem__(self, idx):
#     #     seq = self.sequences[idx]
#     #     label = self.labels[idx]
#     #     onehot = onehot_encode(seq, self.max_len)
#     #     return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)
#     def __getitem__(self, idx):
#         seq = self.sequences[idx]
#         label = self.labels[idx]
#         onehot = onehot_encode(seq, len(seq))  # ← use actual length
#         return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)
    
    
class ProteinDataset(Dataset):
    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

        lengths = [len(seq) for seq in sequences]
        min_len = min(lengths)
        max_len = max(lengths)

        if max_len - min_len > 1:
            raise ValueError(f"Sequences vary too much in length! Found lengths: {set(lengths)}")

        self.seq_len = max_len  # allow minor difference (pad shorter sequences)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]

        onehot = np.zeros((20, self.seq_len), dtype=np.float32)
        for i, aa in enumerate(seq):
            if i >= self.seq_len:  # (safety) but should never happen
                break
            if aa in AA_TO_INDEX:
                onehot[AA_TO_INDEX[aa], i] = 1.0

        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)




In [11]:
class ProteinTransformer(nn.Module):
    def __init__(self, input_dim=20, d_model=128, nhead=4, num_layers=2):
        super(ProteinTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=256, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(d_model, 1),
            nn.Sigmoid()
        )

    def _generate_positional_encoding(self, d_model, length, device):
        pe = torch.zeros(length, d_model, device=device)
        position = torch.arange(0, length, dtype=torch.float, device=device).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, device=device).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # [1, length, d_model]

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B, L, C]
        x = self.embedding(x)   # [B, L, d_model]

        pos_enc = self._generate_positional_encoding(x.size(2), x.size(1), x.device)  # [1, L, d_model]
        x = x + pos_enc  # broadcast over batch

        x = self.transformer_encoder(x)  # [B, L, d_model]
        x = x.permute(0, 2, 1)  # [B, d_model, L]
        return self.classifier(x).squeeze()


In [12]:
# ------------------------- Training & Evaluation -------------------------
def train_transformer_model(gene_name, df, save_dir="auc_results/transformer_results/trained_transformer_models"):
    os.makedirs(save_dir, exist_ok=True)
    df = df[df["Frameshift_Mutation"] == 0].copy()
    df = df[df["Phenotype"].isin(["R", "S"])]
    df["label"] = df["Phenotype"].map({"S": 0, "R": 1})

    sequences = df["Protein_Sequence"].tolist()
    labels = df["label"].values
    max_len = max(len(seq) for seq in sequences)

    dataset = ProteinDataset(sequences, labels)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model = ProteinTransformer()
    model = model.cuda()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    model.train()
    for epoch in range(10):
        losses, all_preds, all_labels = [], [], []
        for X, y in tqdm(dataloader):
            X, y = X.cuda(), y.cuda()
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            all_preds.extend(outputs.detach().cpu().numpy())
            all_labels.extend(y.cpu().numpy())

        auc = roc_auc_score(all_labels, all_preds)
        print(f"Epoch {epoch+1}: Loss={np.mean(losses):.4f}, AUC={auc:.4f}")

    model_path = os.path.join(save_dir, f"{gene_name}_transformer.pt")
    torch.save(model.state_dict(), model_path)
    print(f"Saved model to {model_path}")
    
    result_df = pd.DataFrame({
        "Gene": [gene_name],
        "AUC": [auc]
    })
    result_df.to_csv(f"auc_results/transformer_results/{gene_name}_results.csv", index=False)
    print(f"Saved model to {model_path}")

    return model, max_len



In [13]:
# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GENE_LIST = ['pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']
DATA_DIR = "./data/sequence_data_csv"
CATALOG_DF = pd.read_csv("./data/filtered_variants_output.csv")

In [ ]:
all_results = []

for gene in tqdm(GENE_LIST):
    file_path = f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        model, max_len = train_transformer_model(gene, df)

        # Optional: collect result metrics (e.g., final AUC)
        all_results.append({
            "gene": gene,
            "max_len": max_len,
            # If `train_transformer_model` prints and returns AUC, add that here too
        })
    else:
        print(f"Skipping {gene}: file not found.")

# Save summary
summary_df = pd.DataFrame(all_results)
summary_df.to_csv("auc_results/transformer_results/transformer_auc_scores.csv", index=False)

In [14]:
# Helper: Load dataset and model
def load_gene_data(gene_name):
    df = pd.read_csv(f"{DATA_DIR}/{gene_name}_combined_sequence_data.csv")
    df = df[df["Frameshift_Mutation"] == 0].copy()
    df = df[df["Protein_Sequence"].apply(lambda x: x.count("X") / len(x) < 0.05)].reset_index(drop=True)
    seqs = df["Protein_Sequence"].tolist()
    labels = df["Phenotype"].map({"S": 0.0, "R": 1.0}).values
    max_len = df["Protein_Sequence"].str.len().max()
    return seqs, labels,max_len


In [15]:

def compute_transformer_residue_importance(model, dataloader, device, max_len, save_path=None):
    """
    Compute per-residue attribution scores for a trained Transformer model using input gradient method.

    Args:
        model: Trained nn.Module (Transformer).
        dataloader: DataLoader providing batches of (input, label).
        device: 'cuda' or 'cpu'.
        max_len: Length of the input sequences (padded).
        save_path: Optional path to save importance and labels as .npz.

    Returns:
        importance_matrix: (N, max_len) numpy array of attribution scores.
        labels_array: (N,) numpy array of ground truth labels.
    """

    model.eval()
    model.to(device)
    all_importance_scores = []
    all_labels = []

    for inputs, labels in tqdm(dataloader, desc="Transformer Residue Attribution"):
        inputs = inputs.to(device)                        # shape: (B, 20, max_len)
        labels = labels.to(device)
        inputs.requires_grad_()                          # enable gradient tracking

        outputs = model(inputs)                          # shape: (B,)
        grads = []

        for i in range(len(inputs)):
            model.zero_grad()
            outputs[i].backward(retain_graph=True)

            grad = inputs.grad[i].detach().cpu().numpy()     # shape: (20, max_len)
            score = np.abs(grad).sum(axis=0)                 # → (max_len,)
            grads.append(score)

        all_importance_scores.extend(grads)
        all_labels.extend(labels.cpu().numpy())

        # Reset for next batch
        inputs.requires_grad_(False)
        inputs.grad = None

    importance_matrix = np.array(all_importance_scores)     # shape: (N, max_len)
    labels_array = np.array(all_labels)

    if save_path:
        np.savez(save_path, importance=importance_matrix, labels=labels_array)
        print(f"Saved attribution results to: {save_path}")

    return importance_matrix, labels_array


In [ ]:
WHO_catalog_df = CATALOG_DF  # for clarity

all_results = []

for gene_name in GENE_LIST:
    print(f"\n===== {gene_name} =====")

    # Load sequence data and labels
    seqs, labels, max_len = load_gene_data(gene_name)
    dataset = ProteinDataset(seqs, labels, max_len)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

    # Load trained Transformer model
    model_path = f"trained_transformer_models/{gene_name}_transformer.pt"
    model = ProteinTransformer(input_dim=20, max_len=max_len).to(DEVICE)
    model.load_state_dict(torch.load(model_path))

    # Compute input-gradient based importance
    imp_matrix, labels_arr = compute_transformer_residue_importance(
        model, dataloader, device=DEVICE, max_len=max_len,
        save_path=f"residue_importance/{gene_name}_transformer_residue_importance_global.npz"
    )


### LOO attribution

In [16]:
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from torch.cuda.amp import autocast

# -------------------------
# LOO Attribution -R + Subsampled S for LOO Attribution
# -------------------------
@torch.no_grad()
def compute_patchwise_loo_attribution_fp16(model, input_tensor, device="cuda", patch_size=5):
    model.eval()
    input_tensor = input_tensor.to(device)
    input_tensor = input_tensor.clone()

    original_output = model(input_tensor).item()
    _, _, L = input_tensor.shape
    deltas = np.zeros(L)

    for start in range(0, L, patch_size):
        end = min(start + patch_size, L)
        masked_input = input_tensor.clone()
        masked_input[0, :, start:end] = 0.0

        with autocast():
            masked_output = model(masked_input).float().item()

        delta = original_output - masked_output
        deltas[start:end] = delta / (end - start)

    return deltas



In [ ]:
# -------------------------
# Main Loop per Gene
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gene_list = ['rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB','embC','gyrB','gyrA','ethA','ethR']

for gene in tqdm(gene_list, desc="LOO Attribution"):
    model = ProteinTransformer()
    model.load_state_dict(torch.load(f"auc_results/transformer_results/trained_transformer_models/{gene}_transformer.pt"))
    model.to(device)

    # -------------------------
    # Filter and Sample R + S
    # -------------------------
    df = pd.read_csv(f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv")
    df = df[df["Frameshift_Mutation"] == 0]
    df = df[df["Phenotype"].isin(["R", "S"])]

    df_r = df[df["Phenotype"] == "R"]
    df_s = df[df["Phenotype"] == "S"].sample(n=min(500, len(df[df["Phenotype"] == "S"])), random_state=42)

    df_filtered = pd.concat([df_r, df_s]).reset_index(drop=True)
    print("gene name:", gene)
    print("number of r sequences", len(df_r))
    print("number of S sequences", len(df_s))
    print("total number of sequences selected for loo attribution", len(df_filtered))

    sequences = df_filtered["Protein_Sequence"].tolist()
    labels = df_filtered["Phenotype"].map({"S": 0, "R": 1}).tolist()

    max_len = max(len(seq) for seq in sequences)
    dataset = ProteinDataset(sequences, labels)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

    all_loo_scores = []

    for i, (input_tensor, label) in enumerate(tqdm(dataloader, desc=f"{gene} LOO")):
        loo_score = compute_patchwise_loo_attribution_fp16(model, input_tensor, device, patch_size=5)
        all_loo_scores.append(loo_score)

    loo_array = np.array(all_loo_scores)
    label_array = np.array(labels)

    # Save all LOO scores
    np.savez(f"residue_importance/transformer/npz_files/{gene}_transformer_loo_residue_importance.npz",
             importance=loo_array, labels=label_array)


    # Save mean LOO attribution (over R sequences only)
    mean_loo = loo_array[label_array == 1].mean(axis=0)
    importance_df = pd.DataFrame({
        "Residue_Position": list(range(len(mean_loo))),
        "LOO_Importance": mean_loo
    })

    # Print top 5 most important positions
    print(f"\nTop 5 positions for {gene} by LOO importance:")
    print(importance_df.sort_values("LOO_Importance", ascending=False).head(5))

    # Save
    importance_df.to_csv(f"residue_importance/transformer/{gene}_transformer_residue_importance_loo.csv", index=False)

    print(f"Saved LOO attribution for {gene}")


In [11]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------
# Constants and setup
# -------------------------------
GENE_LIST = ['rpoB', 'rpsL', 'tlyA', 'pncA', 'eis', 'gid', 'katG', 'inhA',
             'embA', 'embB', 'embC', 'gyrB', 'gyrA', 'ethA', 'ethR']
K_VALUES = [10]
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
IMPORTANCE_DIR = Path("residue_importance/transformer")
CATALOG_PATH = Path("./data/filtered_variants_output.csv")
OUTPUT_PATH = Path("residue_importance/transformer_k10.csv")


# -------------------------------
# Load WHO Catalog
# -------------------------------
def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog


# -------------------------------
# Evaluation per Gene
# -------------------------------
def evaluate_gene(gene_name, catalog_df):
    results = []
    csv_path = IMPORTANCE_DIR / f"{gene_name}_transformer_residue_importance_loo.csv"
    if not csv_path.exists():
        print(f"Skipping {gene_name}: missing CSV file.")
        return results

    imp_df = pd.read_csv(csv_path).sort_values("LOO_Importance", ascending=False)

    # Subset WHO variants to this gene
    variants_df = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if variants_df.empty:
        print(f"Skipping {gene_name}: no WHO variants.")
        return []

    true_positions = set(variants_df["aa_pos_0idx"])
    total_actual_positives = len(true_positions)

    for k in K_VALUES:
        topk = imp_df.head(k)
        top_positions = set(topk["Residue_Position"])
        true_positives = len(top_positions.intersection(true_positions))

        precision = true_positives / len(top_positions) if top_positions else 0.0
        recall = true_positives / total_actual_positives if total_actual_positives > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall + 1e-8)

        matched_df = variants_df[variants_df["aa_pos_0idx"].isin(top_positions)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()
        identified_variants_str = ", ".join(identified_variants) if identified_variants else "None"

        results.append({
            "gene": gene_name,
            "Total_Resistance_Positions": total_actual_positives,
            "k": k,
            "TP": true_positives,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "identified_variants": identified_variants_str
        })

    return results


# -------------------------------
# Main
# -------------------------------
catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)
all_results = []

for gene_name in GENE_LIST:
    all_results.extend(evaluate_gene(gene_name, catalog_df))

summary_df = pd.DataFrame(all_results)
summary_df.to_csv(OUTPUT_PATH, index=False)
print(f"✓ Saved Top-k Precision/Recall summary to: {OUTPUT_PATH}")


Skipping eis: no WHO variants.
Skipping embA: no WHO variants.
Skipping embC: no WHO variants.
Skipping ethR: no WHO variants.
✓ Saved Top-k Precision/Recall summary to: residue_importance/transformer_k10.csv


In [ ]:
# Sort by 'gene' ascending and 'precision' descending
df_sorted = final_df.sort_values(by=['Gene', 'Precision'], ascending=[True, False])

# Now, for each gene, pick the first row (highest precision per gene)
best_per_gene = df_sorted.drop_duplicates(subset=['Gene'], keep='first').reset_index(drop=True)

print(best_per_gene)

In [3]:
best_per_gene.to_csv("residue_importance/transformer_pr_overlap_summary.csv", index=False)

## significance testing

In [17]:

import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast

def train_eval_transformer_kfold(gene_name, df, n_splits=5, save_dir="auc_results/transformer_results"):
    df = df[df["Frameshift_Mutation"] == 0].copy()
    df = df[df["Phenotype"].isin(["R", "S"])]
    df["label"] = df["Phenotype"].map({"S": 0, "R": 1})
    sequences = df["Protein_Sequence"].tolist()
    labels = df["label"].values
    max_len = max(len(seq) for seq in sequences)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    results = []

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    for fold, (train_idx, test_idx) in enumerate(skf.split(sequences, labels), 1):
        print(f"[{gene_name}] Fold {fold}")
        fold_dir = os.path.join(save_dir, gene_name, f"fold_{fold}")
        os.makedirs(fold_dir, exist_ok=True)

        X_train = [sequences[i] for i in train_idx]
        y_train = labels[train_idx]
        X_test = [sequences[i] for i in test_idx]
        y_test = labels[test_idx]

        train_loader = DataLoader(ProteinDataset(X_train, y_train), batch_size=32, shuffle=True)
        test_loader = DataLoader(ProteinDataset(X_test, y_test), batch_size=1, shuffle=False)

        model = ProteinTransformer().to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.BCELoss()

        model.train()
        for epoch in range(10):
            for X, y in train_loader:
                X, y = X.to(device), y.to(device)
                optimizer.zero_grad()
                outputs = model(X)
                loss = criterion(outputs, y)
                loss.backward()
                optimizer.step()

        # Save model
        model_path = os.path.join(fold_dir, "model.pt")
        torch.save(model.state_dict(), model_path)

        # Evaluate AUC
        model.eval()
        preds, truths = [], []
        with torch.no_grad():
            for X, y in test_loader:
                outputs = model(X.to(device)).cpu()
                # preds.extend(outputs.numpy())
                preds.extend(outputs.flatten().numpy())

                truths.extend(y.numpy())
        auc = roc_auc_score(truths, preds)
        pd.DataFrame({"Fold": [fold], "AUC": [auc]}).to_csv(os.path.join(fold_dir, "auc.csv"), index=False)

        # Compute LOO per test sequence
        all_loo = []
        for X, _ in test_loader:
            score = compute_patchwise_loo_attribution_fp16(model, X, device=device)
            all_loo.append(score)
        loo_array = np.array(all_loo)
        np.save(os.path.join(fold_dir, "loo.npy"), loo_array)

        # Save mean LOO
        mean_loo = loo_array.mean(axis=0)
        pd.DataFrame({
            "Residue_Position": list(range(len(mean_loo))),
            "LOO_Importance": mean_loo
        }).to_csv(os.path.join(fold_dir, "loo_importance.csv"), index=False)

        results.append({
            "Gene": gene_name,
            "Fold": fold,
            "AUC": auc
        })

    summary_df = pd.DataFrame(results)
    summary_df.to_csv(os.path.join(save_dir, f"{gene_name}_fold_auc_summary.csv"), index=False)
    return summary_df


In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# Assumed to be already defined and imported
# from your_module import train_eval_transformer_kfold

# GENE_LIST = [
#     "rpoB", "katG", "inhA", "embA", "embB", "embC",
#     "pncA", "gyrA", "gyrB", "ethA", "ethR", "eis", "gid", "rpsL", "tlyA"
# ]

GENE_LIST = ["embB"]

def run_all_transformer_folds():
    summary_dfs = []
    for gene in tqdm(GENE_LIST, desc="Running Transformer K-Folds"):
        csv_path = f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv"
        if not os.path.exists(csv_path):
            print(f"[{gene}] Skipping: file not found.")
            continue
        df = pd.read_csv(csv_path)
        summary_df = train_eval_transformer_kfold(gene, df)
        summary_dfs.append(summary_df)

    final_summary = pd.concat(summary_dfs, ignore_index=True)
    final_summary.to_csv("auc_results/transformer_results/transformer_all_fold_auc_summary.csv", index=False)
    print("All folds complete. Results saved to transformer_all_fold_auc_summary.csv")


run_all_transformer_folds()


Running Transformer K-Folds:   0%|          | 0/1 [00:00<?, ?it/s]

[embB] Fold 1
[embB] Fold 2


## significance testing - PR - single gene drugs

In [7]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# Constants
GENES = ["katG", "inhA", "ethA", "ethR", "embB","gid","rpsL","tlyA","eis", "pncA", "gyrA", "gyrB","rpoB"]
CATALOG_PATH = "./data/filtered_variants_output.csv"
PROTEIN_REF_PATH = "./data/catalog/protein_sequences.csv"
LOO_DIR = "./statistical_test_results/prediction_task/transformer_results"
OUTPUT_DIR = "./statistical_test_results/PR_task/transformer_results"
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']


def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog
catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)

# Load protein lengths
ref_df = pd.read_csv(PROTEIN_REF_PATH)
ref_lengths = dict(zip(ref_df["gene"], ref_df["protein_sequence"].str.len()))



def load_mean_loo(gene, fold):
    path = os.path.join(LOO_DIR, gene, f"fold_{fold}", "loo.npy")
    return np.load(path).mean(axis=0)

def evaluate_topk_precision_recall_loocnn(gene, scores, k_vals, catalog_df):
    gene_catalog = catalog_df[catalog_df["gene"].str.lower() == gene.lower()]
    if gene_catalog.empty:
        print(f"[{gene}] Skipping: no confirmed WHO variants.")
        return []

    confirmed_positions = set(gene_catalog["aa_pos_0idx"])
    total_actual = len(confirmed_positions)

    imp_df = pd.DataFrame({
        "Residue_Position": np.arange(len(scores)),
        "Importance": scores
    }).sort_values("Importance", ascending=False)
    print(imp_df.head(5))

    rows = []
    for k in k_vals:
        top_k_pos = set(imp_df.head(k)["Residue_Position"])
        true_positives = len(top_k_pos & confirmed_positions)

        precision = true_positives / k
        recall = true_positives / total_actual if total_actual > 0 else 0
        f1 = 2 * precision * recall / (precision + recall + 1e-8)

        matched_df = gene_catalog[gene_catalog["aa_pos_0idx"].isin(top_k_pos)]
        identified_variants = matched_df.drop_duplicates("aa_pos_0idx")["variant"].tolist()

        rows.append({
            "gene": gene,
            "TP": true_positives,
            "Total_Resistance_Positions": total_actual,
            "precision": precision,
            "recall": recall,
            "F1": f1,
            "k": k,
            "identified_variants": ", ".join(identified_variants) if identified_variants else "None"
        })
        print(rows[:5])

    return rows

def run_single_gene_lootransformer(gene_list=GENES, k_vals=[10]):
    all_rows = []
    for gene in gene_list:
        print(f"\n=== Evaluating {gene} ===")
        for fold in range(1, 6):
            print(f"[{gene}] Fold {fold}")
            scores = load_mean_loo(gene, fold)

            rows = evaluate_topk_precision_recall_loocnn(
                gene=gene,
                scores=scores,
                k_vals=k_vals,
                catalog_df=catalog_df
            )

            for r in rows:
                r.update({
                    "model": "transformer",
                    "fold": fold,
                    "gene_set": gene
                })
            all_rows.extend(rows)

        out_df = pd.DataFrame(all_rows)
        out_df.to_csv(os.path.join(OUTPUT_DIR, f"{gene}_transformer_pr_per_fold.csv"), index=False)

    print("All single-gene transformer PR evaluations completed.")
    return all_rows

# Run the evaluation
run_single_gene_lootransformer()


=== Evaluating katG ===
[katG] Fold 1
    Residue_Position  Importance
39                39    0.000022
38                38    0.000022
37                37    0.000022
36                36    0.000022
35                35    0.000022
[{'gene': 'katG', 'TP': 0, 'Total_Resistance_Positions': 2, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'k': 10, 'identified_variants': 'None'}]
[katG] Fold 2
     Residue_Position  Importance
246               246   -0.000007
249               249   -0.000007
248               248   -0.000007
247               247   -0.000007
245               245   -0.000007
[{'gene': 'katG', 'TP': 0, 'Total_Resistance_Positions': 2, 'precision': 0.0, 'recall': 0.0, 'F1': 0.0, 'k': 10, 'identified_variants': 'None'}]
[katG] Fold 3
     Residue_Position  Importance
315               315    0.000083
319               319    0.000083
318               318    0.000083
317               317    0.000083
316               316    0.000083
[{'gene': 'katG', 'TP': 0, 'Total_Res

FileNotFoundError: [Errno 2] No such file or directory: './statistical_test_results/prediction_task/transformer_results/embB/fold_1/loo.npy'